In [6]:
import os
import pandas as pd
import numpy as np
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score, adjusted_rand_score, normalized_mutual_info_score

print("✅ All imports successful!\n")

# Load the data
df_no_pca = pd.read_csv('../data/preprocessed_data_no_pca.csv')
df_pca = pd.read_csv('../data/preprocessed_data_with_pca.csv')

print("=== DATA LOADED ===")
print(f"Original data shape: {df_no_pca.shape}")
print(f"PCA data shape: {df_pca.shape}")
print(f"\nOriginal data columns: {df_no_pca.columns.tolist()}")
print(f"\nFirst 3 rows of original data:")
print(df_no_pca.head(3))

✅ All imports successful!

=== DATA LOADED ===
Original data shape: (1348, 5)
PCA data shape: (1348, 2)

Original data columns: ['variance', 'skewness', 'curtosis', 'entropy', 'outlier_flag']

First 3 rows of original data:
   variance  skewness  curtosis   entropy  outlier_flag
0  1.109709  1.151820 -0.975529  0.346132     -0.269062
1  1.432683  1.066810 -0.894937 -0.140707     -0.269062
2  1.195109 -0.775147  0.118015  0.611558     -0.269062


In [7]:
print("Unique values in outlier_flag:")
print(df_no_pca['outlier_flag'].unique())

print("\nValue counts:")
print(df_no_pca['outlier_flag'].value_counts())

Unique values in outlier_flag:
[-0.26906243  3.71660959]

Value counts:
outlier_flag
-0.269062    1257
 3.716610      91
Name: count, dtype: int64


In [8]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score, adjusted_rand_score, normalized_mutual_info_score

# Features (first 4 columns)
X = df_no_pca[['variance', 'skewness', 'curtosis', 'entropy']]

# Ground truth (already scaled)
y_true = df_no_pca['outlier_flag']

# Scaled the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
labels = kmeans.fit_predict(X_scaled)

df_no_pca['Cluster'] = labels

print("=== K-MEANS CLUSTERING COMPLETE ===")
print(f"Cluster distribution:\n{df_no_pca['Cluster'].value_counts()}")
print(f"\nGround truth distribution:\n{df_no_pca['outlier_flag'].value_counts()}")


print("\n" + "="*50)
print("INTERNAL METRICS (cluster quality only)")
print("="*50)
sil = silhouette_score(X_scaled, labels)
dbi = davies_bouldin_score(X_scaled, labels)
ch = calinski_harabasz_score(X_scaled, labels)

print(f"Silhouette Score:        {sil:.4f}  (higher is better, -1 to 1)")
print(f"Davies-Bouldin Index:    {dbi:.4f}  (lower is better)")
print(f"Calinski-Harabasz Index: {ch:.2f}   (higher is better)")

print("\n" + "="*50)
print("EXTERNAL METRICS (comparing to ground truth)")
print("="*50)
ari = adjusted_rand_score(y_true, labels)
nmi = normalized_mutual_info_score(y_true, labels)

print(f"Adjusted Rand Index (ARI): {ari:.4f}  (1.0 = perfect agreement)")
print(f"Normalized Mutual Info:    {nmi:.4f}  (1.0 = perfect agreement)")

# Check cluster alignment with ground truth
print("\n" + "="*50)
print("CONFUSION MATRIX (Cluster vs Ground Truth)")
print("="*50)
crosstab = pd.crosstab(df_no_pca['outlier_flag'], df_no_pca['Cluster'], 
                       normalize='all', margins=True)
print(crosstab)

=== K-MEANS CLUSTERING COMPLETE ===
Cluster distribution:
Cluster
1    688
0    660
Name: count, dtype: int64

Ground truth distribution:
outlier_flag
-0.269062    1257
 3.716610      91
Name: count, dtype: int64

INTERNAL METRICS (cluster quality only)
Silhouette Score:        0.3280  (higher is better, -1 to 1)
Davies-Bouldin Index:    1.2042  (lower is better)
Calinski-Harabasz Index: 788.77   (higher is better)

EXTERNAL METRICS (comparing to ground truth)
Adjusted Rand Index (ARI): -0.0001  (1.0 = perfect agreement)
Normalized Mutual Info:    0.0060  (1.0 = perfect agreement)

CONFUSION MATRIX (Cluster vs Ground Truth)
Cluster              0         1       All
outlier_flag                              
-0.269062     0.465875  0.466617  0.932493
 3.71661      0.023739  0.043769  0.067507
 All          0.489614  0.510386  1.000000


C:\Users\Ndanu\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\metrics\cluster\_supervised.py:69: UserWarning: Clustering metrics expects discrete values but received continuous values for label, and binary values for target
  warnings.warn(msg, UserWarning)
C:\Users\Ndanu\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\metrics\cluster\_supervised.py:69: UserWarning: Clustering metrics expects discrete values but received continuous values for label, and binary values for target
  warnings.warn(msg, UserWarning)


In [9]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score, adjusted_rand_score, normalized_mutual_info_score
import pandas as pd
import numpy as np

# Load data
df_no_pca = pd.read_csv('../data/preprocessed_data_no_pca.csv')
df_pca = pd.read_csv('../data/preprocessed_data_with_pca.csv')

# Features
X = df_no_pca[['variance', 'skewness', 'curtosis', 'entropy']]

# Convert ground truth to integers (0 and 1)
# Original: -0.269062 = Authentic (0), 3.716610 = Counterfeit (1)
y_true = (df_no_pca['outlier_flag'] > 0).astype(int)

print("=== Ground truth converted ===")
print(f"0 (Authentic): {(y_true==0).sum()} samples")
print(f"1 (Counterfeit): {(y_true==1).sum()} samples")

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Run K-Means
kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
labels = kmeans.fit_predict(X_scaled)

print("\n=== K-MEANS RESULTS ===")
print(f"Cluster distribution:\n{pd.Series(labels).value_counts()}")

# Internal metrics
print("\n" + "="*50)
print("INTERNAL METRICS")
print("="*50)
sil = silhouette_score(X_scaled, labels)
dbi = davies_bouldin_score(X_scaled, labels)
ch = calinski_harabasz_score(X_scaled, labels)

print(f"Silhouette Score:        {sil:.4f}  (higher is better)")
print(f"Davies-Bouldin Index:    {dbi:.4f}  (lower is better)")
print(f"Calinski-Harabasz Index: {ch:.2f}   (higher is better)")

# External metrics
print("\n" + "="*50)
print("EXTERNAL METRICS (vs Ground Truth)")
print("="*50)
ari = adjusted_rand_score(y_true, labels)
nmi = normalized_mutual_info_score(y_true, labels)

print(f"Adjusted Rand Index (ARI): {ari:.4f}  (1.0 = perfect)")
print(f"Normalized Mutual Info:    {nmi:.4f}  (1.0 = perfect)")

# Confusion matrix
print("\n" + "="*50)
print("CONFUSION MATRIX")
print("="*50)
cm = pd.crosstab(y_true, labels, rownames=['True Class'], colnames=['Predicted Cluster'])
print(cm)

# Accuracy interpretation
matches = ((y_true == 0) & (labels == 0)).sum() + ((y_true == 1) & (labels == 1)).sum()
total = len(y_true)
print(f"\nCluster alignment accuracy: {matches/total:.2%}")

=== Ground truth converted ===
0 (Authentic): 1257 samples
1 (Counterfeit): 91 samples

=== K-MEANS RESULTS ===
Cluster distribution:
1    688
0    660
Name: count, dtype: int64

INTERNAL METRICS
Silhouette Score:        0.3280  (higher is better)
Davies-Bouldin Index:    1.2042  (lower is better)
Calinski-Harabasz Index: 788.77   (higher is better)

EXTERNAL METRICS (vs Ground Truth)
Adjusted Rand Index (ARI): -0.0001  (1.0 = perfect)
Normalized Mutual Info:    0.0060  (1.0 = perfect)

CONFUSION MATRIX
Predicted Cluster    0    1
True Class                 
0                  628  629
1                   32   59

Cluster alignment accuracy: 50.96%


In [10]:
# PCA Data
X_pca = df_pca.values  # PC1, PC2

kmeans_pca = KMeans(n_clusters=2, random_state=42, n_init=10)
labels_pca = kmeans_pca.fit_predict(X_pca)

print("\n" + "="*50)
print("PCA-REDUCED DATA RESULTS")
print("="*50)
print(f"Cluster distribution:\n{pd.Series(labels_pca).value_counts()}")

# Internal metrics
sil_pca = silhouette_score(X_pca, labels_pca)
dbi_pca = davies_bouldin_score(X_pca, labels_pca)
ch_pca = calinski_harabasz_score(X_pca, labels_pca)

print(f"\nSilhouette Score:        {sil_pca:.4f}")
print(f"Davies-Bouldin Index:    {dbi_pca:.4f}")
print(f"Calinski-Harabasz Index: {ch_pca:.2f}")

# External metrics
ari_pca = adjusted_rand_score(y_true, labels_pca)
nmi_pca = normalized_mutual_info_score(y_true, labels_pca)

print(f"\nAdjusted Rand Index (ARI): {ari_pca:.4f}")
print(f"Normalized Mutual Info:    {nmi_pca:.4f}")

# Confusion matrix
print("\n" + "="*50)
print("CONFUSION MATRIX (PCA)")
print("="*50)
cm_pca = pd.crosstab(y_true, labels_pca, rownames=['True Class'], colnames=['Predicted Cluster'])
print(cm_pca)

matches_pca = ((y_true == 0) & (labels_pca == 0)).sum() + ((y_true == 1) & (labels_pca == 1)).sum()
print(f"\nCluster alignment accuracy: {matches_pca/total:.2%}")


PCA-REDUCED DATA RESULTS
Cluster distribution:
1    677
0    671
Name: count, dtype: int64

Silhouette Score:        0.3941
Davies-Bouldin Index:    1.0104
Calinski-Harabasz Index: 803.92

Adjusted Rand Index (ARI): 0.0011
Normalized Mutual Info:    0.0067

CONFUSION MATRIX (PCA)
Predicted Cluster    0    1
True Class                 
0                  639  618
1                   32   59

Cluster alignment accuracy: 51.78%


In [11]:
# Comparison table
comparison_table = pd.DataFrame({
    'Metric': ['Silhouette Score ↑', 'Davies-Bouldin Index ↓', 'Calinski-Harabasz ↑', 
               'Adjusted Rand Index (ARI) ↑', 'Normalized Mutual Info (NMI) ↑', 
               'Cluster Alignment Accuracy'],
    'Original Data': [f"{sil:.4f}", f"{dbi:.4f}", f"{ch:.2f}", f"{ari:.4f}", f"{nmi:.4f}", f"{matches/total:.2%}"],
    'PCA-Reduced Data': [f"{sil_pca:.4f}", f"{dbi_pca:.4f}", f"{ch_pca:.2f}", f"{ari_pca:.4f}", f"{nmi_pca:.4f}", f"{matches_pca/total:.2%}"]
})

print("\n" + "="*60)
print("COMPREHENSIVE METRICS TABLE (For Report)")
print("="*60)
print(comparison_table.to_string(index=False))


comparison_table.to_csv('../data/kmeans_metrics_comparison.csv', index=False)
print("\n✅ Saved to: ../data/kmeans_metrics_comparison.csv")


COMPREHENSIVE METRICS TABLE (For Report)
                        Metric Original Data PCA-Reduced Data
            Silhouette Score ↑        0.3280           0.3941
        Davies-Bouldin Index ↓        1.2042           1.0104
           Calinski-Harabasz ↑        788.77           803.92
   Adjusted Rand Index (ARI) ↑       -0.0001           0.0011
Normalized Mutual Info (NMI) ↑        0.0060           0.0067
    Cluster Alignment Accuracy        50.96%           51.78%

✅ Saved to: ../data/kmeans_metrics_comparison.csv
